In [1]:
# setup environment variables
import dotenv 

dotenv.load_dotenv("./shell_scripts/spotify_variables.env")

True

In [2]:
from playlist_tools import PlaylistTools
import load_playlists

In [12]:
# load the library data
library = load_playlists.load_base_playlists()
library.check_missing_songs_in_playlistsets_not_in_library()

Found exactly 1 playlist with name BASE_LIBRARY
loaded playlist BASE_LIBRARY. len tracks: 987
init called for type <class 'base_playlists.BasePlaylistSetStyle'>
Found exactly 1 playlist with name BASE_BLUES
loaded playlist BASE_BLUES. len tracks: 33
init of BasePlaylistSetEnergy called
init called for type <class 'base_playlists.BasePlaylistSetEnergy'>
Found exactly 1 playlist with name BASE_ENERGY_LOW
loaded playlist BASE_ENERGY_LOW. len tracks: 398
Found exactly 1 playlist with name BASE_ENERGY_MEDIUM
loaded playlist BASE_ENERGY_MEDIUM. len tracks: 387
Found exactly 1 playlist with name BASE_ENERGY_HIGH
loaded playlist BASE_ENERGY_HIGH. len tracks: 198
Creating or cleaning playlist MISSING_ENERGY for user dulinn
Creating or cleaning playlist DUPLICATE_ENERGY for user dulinn
init called for type <class 'base_playlists.BasePlaylistSetBPM'>
Found exactly 1 playlist with name BASE_BPM_BELOW_70
loaded playlist BASE_BPM_BELOW_70. len tracks: 4
Found exactly 1 playlist with name BASE_BPM_70

In [4]:
from pprint import pprint
from datetime import timedelta, datetime

load_playlists.generate_playlists(library)

Creating or cleaning playlist WCS Large for user dulinn
Creating or cleaning playlist WCS Favorites for user dulinn
Creating or cleaning playlist WCS Last 6 Months for user dulinn
Creating or cleaning playlist WCS Medium and Low Energy for user dulinn
Creating or cleaning playlist WCS Walk 100-114 BPM for user dulinn
Creating or cleaning playlist WCS Walk 115-124 BPM for user dulinn


In [5]:
#library = load_playlists.load_base_library()
#load_playlists.load_base_style(library)

In [22]:
from playlist_tools import PlaylistTools
from pprint import pprint
from datetime import timedelta, datetime
from collections import Counter
playlist_start_time = datetime(year=2025, month=11, day=15, hour=21, minute=35)

playlist_tools = PlaylistTools()
playlist_id = "674QNCxZZ1QFGX1Gq7mvsm"
debug_playlist = False

def indicator_str(low, high, length, value):
    value=min(value, high)
    value=max(value, low)
    value=value-low
    low_to_high = high-low
    value_per_dash = low_to_high/length
    num_dashes = int(value/value_per_dash)
    return " "*(length-num_dashes)+"*"*num_dashes


track_stats = {}
missing_bpm = []
missing_energy = []
missing_like = []
missing_in_base_library = []
playlist_time_seconds = 0
previous_bpm = 0
previous_energy = 0
for i, track in enumerate(playlist_tools.get_and_iterate_tracks(playlist_id)):
    track_id = track['id']
    track_start_time= playlist_start_time + timedelta(seconds=playlist_time_seconds)
    track_stat = library.get_track_statistics(track_id)
    track_stats[track_id] = track_stat
    if "bpm" not in track_stat:
        missing_bpm.append(track_id)
    if "energy" not in track_stat:
        missing_energy.append(track_id)
    if "like" not in track_stat:
        missing_like.append(track_id)
    if track_id not in library.library_playlist.track_ids:
        missing_in_base_library.append(track_id)

    bpm = track_stat.get("bpm", 0)
    bpm_str = indicator_str(70, 120, 10, bpm) if bpm != 0 else "-"*10
    bpm_number_str = f"{bpm:3}" if bpm != 0 else "---"
    energy = track_stat.get("energy", 0)
    energy_str = indicator_str(0, 3, 3, energy) if energy != 0 else "-"*3
    like = track_stat.get("like", 0)
    like_str = indicator_str(0, 3, 3, like) if like != 0 else "-"*3
    attention_indicator = "   "
    if bpm == 0 or energy == 0:
        attention_indicator = "!1 "
    if previous_bpm != 0 and bpm != 0 and abs(bpm - previous_bpm) > 20:
        attention_indicator = "!2 "
    if previous_energy != 0 and energy != 0 and abs(energy - previous_energy) >= 2:
        attention_indicator = "!3 "
    if debug_playlist:
        print(f"{attention_indicator}{i+1:02} TIME: {timedelta(seconds=playlist_time_seconds)} BPM: {bpm_str} ({bpm_number_str}) ENERGY: {energy_str} - {playlist_tools.track_to_str(track)}")
    else:
        print(f"{datetime.strftime(track_start_time, '%H:%M:%S')} BPM: {bpm_number_str} ENERGY: {energy} {playlist_tools.track_to_str(track)}")
    playlist_time_seconds+=int(track['duration_ms']/1000)
    previous_bpm = bpm
    previous_energy = energy




21:35:00 BPM: 107 ENERGY: 2 Hot In Herre - Nelly
21:38:48 BPM:  87 ENERGY: 3 Remember the Name (feat. Styles of Beyond) - Fort Minor, Styles Of Beyond
21:42:38 BPM:  97 ENERGY: 3 The Next Episode - Dr. Dre, Snoop Dogg
21:45:19 BPM: 107 ENERGY: 2 Mangos mit Chili - Nina Chuba
21:47:33 BPM: 117 ENERGY: 3 Circus - Britney Spears
21:50:45 BPM: 102 ENERGY: 2 Lonely (with Jonas Brothers) - Diplo, Jonas Brothers
21:53:04 BPM:  92 ENERGY: 2 I Warned Myself - Charlie Puth
21:55:43 BPM:  92 ENERGY: 2 Sad Girl - Charlotte Cardin
21:58:58 BPM:  87 ENERGY: 2 Looks Like Me - Dean Lewis
22:02:08 BPM:  97 ENERGY: 2 Memory Lane - Zara Larsson
22:05:20 BPM:  97 ENERGY: 2 Mad - Anthony Hamilton
22:09:02 BPM: 102 ENERGY: 2 Float On - Phil Good
22:11:58 BPM:  92 ENERGY: 2 Demons - Imagine Dragons
22:14:55 BPM:  92 ENERGY: 3 Your Idol - Saja Boys, Andrew Choi, Neckwav, Danny Chung, KEVIN WOO, samUIL Lee, KPop Demon Hunters Cast
22:18:06 BPM:  97 ENERGY: 2 Let the Rhythm Just - The Polish Ambassador, Mr. Lif

In [11]:
# create playlists for missing stats
missing_bpm_playlist = playlist_tools.create_or_clean_playlist("Missing BPM temp", quiet=True)
missing_energy_playlist = playlist_tools.create_or_clean_playlist("Missing Energy temp", quiet=True)
missing_like_playlist = playlist_tools.create_or_clean_playlist("Missing Like temp", quiet=True)
missing_in_base_library_playlist = playlist_tools.create_or_clean_playlist("Missing in Base Library temp", quiet=True)
playlist_tools.add_tracks_to_playlist(missing_bpm_playlist, missing_bpm)
playlist_tools.add_tracks_to_playlist(missing_energy_playlist, missing_energy)
playlist_tools.add_tracks_to_playlist(missing_like_playlist, missing_like)
playlist_tools.add_tracks_to_playlist(missing_in_base_library_playlist, missing_in_base_library)

Creating or cleaning playlist Missing BPM temp for user dulinn
Creating or cleaning playlist Missing Energy temp for user dulinn
Creating or cleaning playlist Missing Like temp for user dulinn
Creating or cleaning playlist Missing in Base Library temp for user dulinn


In [14]:
import datetime
playlist_start_time = datetime.datetime(year=2025, month=11, day=15, hour=21, minute=30)


playlist_time_seconds = 0
for i, track in enumerate(playlist_tools.get_and_iterate_tracks(playlist_id)):
    track_start_time= playlist_start_time + timedelta(seconds=playlist_time_seconds)
    playlist_time_seconds+=int(track['duration_ms']/1000)
    print(f"{datetime.datetime.strftime(track_start_time, '%H:%M:%S')}: {playlist_tools.track_to_str(track)}")

21:30:00: Hot In Herre - Nelly
21:33:48: The Next Episode - Dr. Dre, Snoop Dogg
21:36:29: Remember the Name (feat. Styles of Beyond) - Fort Minor, Styles Of Beyond
21:40:19: Rocketeer - Far East Movement, Ryan Tedder, Ruff Loaderz
21:43:50: Circus - Britney Spears
21:47:02: Lonely (with Jonas Brothers) - Diplo, Jonas Brothers
21:49:21: Mangos mit Chili - Nina Chuba
21:51:35: I Warned Myself - Charlie Puth
21:54:14: Sad Girl - Charlotte Cardin
21:57:29: Looks Like Me - Dean Lewis
22:00:39: Memory Lane - Zara Larsson
22:03:51: Mad - Anthony Hamilton
22:07:33: Float On - Phil Good
22:10:29: Demons - Imagine Dragons
22:13:26: Your Idol - Saja Boys, Andrew Choi, Neckwav, Danny Chung, KEVIN WOO, samUIL Lee, KPop Demon Hunters Cast
22:16:37: Let the Rhythm Just - The Polish Ambassador, Mr. Lif, Ayla Nereo
22:22:32: Big Energy - Latto
22:25:24: Still Hot - Nic D, Connor Price
22:27:34: Miami - Will Smith
22:30:51: SexyBack - Ilkan Gunuc, Clara Stegall
22:33:19: LOL - Mabel
22:36:51: Self Contr